In [ ]:
import matplotlib.pyplot as plt
from algbench import read_as_pandas
from shapely import Polygon
from shapely.plotting import plot_polygon
from tspn_bnb2.misc import init_params
from tspn_bnb2.schemas import AnnotatedInstance

init_params()

In [ ]:
def read_row(row):
    if row["result"]["output_instance"] is None:
        return None
    return {
        "output_instance": AnnotatedInstance.model_validate_json(row["result"]["output_instance"]),
        "annotated_instance": AnnotatedInstance.model_validate_json(
            row["result"]["annotated_instance"]
        ),
        "instance_name": row["parameters"]["args"]["kwargs"]["instance_name"],
        "instance": AnnotatedInstance.model_validate_json(
            row["parameters"]["args"]["kwargs"]["instance_json"]
        ),
        "simplification_time": row["result"]["simplification_time"],
        "remove_holes": row["parameters"]["args"]["alg_params"]["remove_holes"],
        "convex_hull_fill": row["parameters"]["args"]["alg_params"]["convex_hull_fill"],
        "remove_supersites": row["parameters"]["args"]["alg_params"]["remove_supersites"],
    }


benchmark = read_as_pandas("results_simplification_socg_60", read_row)
print("loaded", len(benchmark), "simplifications")

In [ ]:
from shapely.ops import unary_union
from shapely.validation import make_valid


def extract_polygons(geom):
    if geom.is_empty:
        return []

    if geom.geom_type == "Polygon":
        return [geom]

    if geom.geom_type == "MultiPolygon":
        return list(geom.geoms)

    if geom.geom_type == "GeometryCollection":
        return [g for g in geom.geoms if g.geom_type == "Polygon"]

    return []

In [ ]:
n_shown = 10
for _, row in benchmark[benchmark["convex_hull_fill"]].iterrows():
    # n_shown -= 1
    if n_shown == 0:
        break

    fig, axs = plt.subplots(ncols=3, figsize=(6, 3))

    for ax in axs.flat:
        ax.set_axis_off()
        ax.set_aspect("equal")

    for polygon in row["instance"].polygons:
        # assert polygon.is_valid
        plot_polygon(polygon=polygon, ax=axs[0], add_points=False, color="grey")
        plot_polygon(polygon=polygon, ax=axs[1], add_points=False, color="grey")

    for polygon in row["output_instance"].polygons:
        # assert polygon.is_valid, f"{polygon.wkt}"
        plot_polygon(polygon=polygon, ax=axs[2], add_points=False, color="grey")

    input_geom = make_valid(unary_union(row["instance"].polygons)).buffer(0)
    output_geom = make_valid(unary_union(row["output_instance"].polygons)).buffer(0)

    removed = extract_polygons(input_geom.difference(output_geom))
    added = extract_polygons(output_geom.difference(input_geom))

    if not removed and not added:
        continue

    print(
        row["instance_name"], row["convex_hull_fill"], row["remove_holes"], row["remove_supersites"]
    )

    fig.suptitle(row["instance_name"])

    axs[0].set_title("Input")
    axs[1].set_title("Simplified")
    axs[2].set_title("Output")

    for polygon in removed:
        if not isinstance(polygon, Polygon):
            continue
        plot_polygon(polygon=polygon, ax=axs[1], add_points=False, color="red", alpha=0.3)

    for polygon in added:
        if not isinstance(polygon, Polygon):
            continue
        plot_polygon(polygon=polygon, ax=axs[1], add_points=False, color="green", alpha=0.3)

    fig.tight_layout()

    fig.savefig("out/" + row["instance_name"] + ".png", dpi=150, facecolor="white")